# 03 — Model Training
Trains every Level-0 base learner: GNN, TCN, Transformer, XGBoost, Cox PH / DeepSurv (Section A3, Deliverables D4-D7).

This mirrors `demo/run_pipeline.py` Phase 2 exactly. For a faster run reduce the epoch counts below; for the numbers reported in the README, use the defaults (or just run `python -m demo.run_pipeline`).

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
from src.data.synthetic_generator import SupplyChainDataGenerator
gen = SupplyChainDataGenerator(seed=42)
nodes, edges = gen.generate_graph()
ts = gen.generate_time_series(n_days=730)
shipments = gen.generate_shipments(n=3000)

## GNN (HetGAT) — Section A3.1

In [2]:
from src.models.gnn import train_gnn
gnn_model, gnn_heads, gnn_embeddings, gnn_node_ids, gnn_metrics = train_gnn(nodes, edges)
gnn_metrics

  [GNN] epoch   0  loss=14.1726  (cls=6.8568, link=7.3158)


  [GNN] epoch  20  loss=7.7891  (cls=1.9003, link=5.8888)


  [GNN] epoch  40  loss=6.3170  (cls=0.5576, link=5.7593)


  [GNN] epoch  60  loss=5.7435  (cls=0.1185, link=5.6250)


  [GNN] epoch  80  loss=5.6435  (cls=0.0536, link=5.5899)


...

  [GNN] epoch 120  loss=5.1083  (cls=0.0330, link=5.0752)


  [GNN] epoch 140  loss=4.9375  (cls=0.0483, link=4.8892)


  [GNN] epoch 160  loss=4.9205  (cls=0.0550, link=4.8655)


  [GNN] epoch 179  loss=4.6389  (cls=0.0395, link=4.5993)


{'node_classification_accuracy': 0.7727272727272727,
 'supplier_risk_tier_accuracy': 0.8333333333333334,
 'link_prediction_auc': 0.637850667734279}

## TCN — Section A3.2

In [3]:
from src.models.tcn import train_tcn
tcn_model, tcn_scalers, tcn_metrics = train_tcn(ts, value_col='throughput_teu',
    exog_col='port_congestion_index', epochs=15, max_entities=6)
tcn_metrics

  [TCN] epoch   0  pinball_loss=0.2307


  [TCN] epoch   5  pinball_loss=0.1490


  [TCN] epoch  10  pinball_loss=0.1370


  [TCN] epoch  14  pinball_loss=0.1313


{'mape_30d_by_entity': {'Los Angeles': 1.7857921952944629,
  'Manzanillo': 2.1957659226896764,
  'Mumbai (JNPT)': 5.827508460513812,
  'Rotterdam': 2.0061318638900905,
  'Shanghai': 11.40202346446428,
  'Singapore': 22.491908197561948},
 'mape_30d_mean': 7.6181883507357115}

## Transformer — Section A3.3

In [4]:
from src.models.transformer import train_shipment_transformer
transformer_model, transformer_metrics = train_shipment_transformer(shipments, epochs=35)
transformer_metrics

  [Transformer] epoch   0  loss=1.4456


  [Transformer] epoch   4  loss=1.0653


  [Transformer] epoch   8  loss=1.0488


  [Transformer] epoch  12  loss=1.0395


  [Transformer] epoch  16  loss=1.0419


...

  [Transformer] epoch  24  loss=1.0372


  [Transformer] epoch  28  loss=1.0322


  [Transformer] epoch  32  loss=1.0307


  [Transformer] epoch  34  loss=1.0348


{'delay_auc': 0.5978860294117647,
 'delay_brier': 0.0850081667304039,
 'damage_auc': 0.5332051548534608,
 'doc_discrepancy_auc': 0.5604137667512759}

## XGBoost + SHAP — Section A7.1

In [5]:
from src.models.xgboost_model import train_xgboost_default_model, compute_shap_values
xgb_model, xgb_split, xgb_metrics, importances = train_xgboost_default_model(nodes, n_trials=15)
print({k: v for k, v in xgb_metrics.items() if k != 'best_params'})
importances.head(10)

  [XGBoost] Optuna best AUC (val): 0.8333
{'auc': 0.9024390243902439, 'gini': 0.8048780487804879, 'brier': 0.10655906051397324, 'average_precision': 0.34444444444444444}


inventory_turnover            0.091048
working_capital_ratio         0.085684
in_degree                     0.064603
supplier_concentration_hhi    0.062502
current_ratio                 0.062208
ebitda_margin                 0.058453
ccc_days                      0.053310
geopolitical_risk_score       0.053304
clustering_coefficient        0.050939
customer_concentration_hhi    0.050921
dtype: float32

## Survival analysis: Cox PH + DeepSurv — Section A3.4

In [6]:
from src.models.survival import fit_cox_ph, train_deepsurv
cox_model, cox_fin_model, cox_metrics = fit_cox_ph(nodes)
cox_metrics

{'c_index_train': 0.8960261295590637,
 'c_index_test_full': 0.8132530120481928,
 'c_index_test_financial_only': 0.6867469879518072,
 'c_index_improvement': 0.12650602409638556}

In [7]:
deepsurv_model, deepsurv_metrics = train_deepsurv(nodes, epochs=100, n_folds=5)
deepsurv_metrics

  [DeepSurv] fold 1/5  n_events_in_fold=5  c_index=0.7024
  [DeepSurv] fold 2/5  n_events_in_fold=2  c_index=0.5294


  [DeepSurv] fold 3/5  n_events_in_fold=3  c_index=0.5854
  [DeepSurv] fold 4/5  n_events_in_fold=2  c_index=0.3735


  [DeepSurv] fold 5/5  n_events_in_fold=3  c_index=0.1626


{'c_index_cv_mean': 0.47066244893490755,
 'c_index_cv_std': 0.18690741925193924,
 'c_index_per_fold': [0.7024390243902439,
  0.5294117647058824,
  0.5853658536585366,
  0.37349397590361444,
  0.16260162601626016]}